# Full Cat Breed Classification Project - Google Colab

Course: ISB46703 Principle of Artificial Intelligence  
Domain: Animal subspecies  
Dataset collection method: Web crawler  
Target dataset size: 10,000 cat breed images  
Models: ResNet50, DenseNet121, MobileNetV3Small

This all-in-one notebook is designed for Google Colab so files do not go missing between separate runtimes. It will:

1. Create a Google Drive project folder.
2. Use a web crawler tool to collect 10,000 cat breed images.
3. Use five cat breed classes with 2,000 images per class.
4. Clean unreadable images.
5. Create training, validation, and testing dataset splits.
6. Train ResNet50 for 50 epochs.
7. Train DenseNet121 for 50 epochs.
8. Train MobileNetV3Small for 50 epochs.
9. Evaluate accuracy, mAP, training time, parameters, and confusion matrix.
10. Create a final comparison table and conclusion.

Before running model training, enable GPU:

`Runtime > Change runtime type > Hardware accelerator > GPU`

## 1. Install Web Crawler And Import Libraries

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'icrawler'])

import json
import random
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from IPython.display import Image as DisplayImage, display
from PIL import Image, UnidentifiedImageError
from sklearn.metrics import accuracy_score, average_precision_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.applications import DenseNet121, MobileNetV3Small, ResNet50
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as mobilenetv3_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet50_preprocess
from icrawler.builtin import BingImageCrawler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 50
IMAGES_PER_CLASS = 2000
TOTAL_TARGET_IMAGES = 10000
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15

print("TensorFlow version:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))

## 2. Create Google Drive Project Folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/AI-Image-Classification-Project')
RAW_DIR = PROJECT_ROOT / 'raw_crawled_images'
DATASET_DIR = PROJECT_ROOT / 'dataset'
RESULTS_DIR = PROJECT_ROOT / 'results'
GRAPHS_DIR = RESULTS_DIR / 'graphs'
CONFUSION_DIR = RESULTS_DIR / 'confusion_matrices'
MODELS_DIR = PROJECT_ROOT / 'models'

for folder in [PROJECT_ROOT, RAW_DIR, DATASET_DIR, RESULTS_DIR, GRAPHS_DIR, CONFUSION_DIR, MODELS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('Google Drive project folder created at:')
print(PROJECT_ROOT)
print('\nDataset folders will be created at:')
print(DATASET_DIR / 'train')
print(DATASET_DIR / 'validation')
print(DATASET_DIR / 'test')

## 3. Web Crawl 10,000 Cat Breed Images

In [ ]:
SELECTED_CLASSES = {
    'Abyssinian': 'Abyssinian cat breed',
    'Bengal': 'Bengal cat breed',
    'Birman': 'Birman cat breed',
    'Bombay': 'Bombay cat breed',
    'British_Shorthair': 'British Shorthair cat breed',
}

print('Selected classes:', list(SELECTED_CLASSES.keys()))
print('Images per class:', IMAGES_PER_CLASS)
print('Target total images:', TOTAL_TARGET_IMAGES)

In [ ]:
def crawl_images_for_class(class_name, search_query, max_images=IMAGES_PER_CLASS):
    class_dir = RAW_DIR / class_name
    class_dir.mkdir(parents=True, exist_ok=True)

    existing_images = list(class_dir.glob('*'))
    if len(existing_images) >= max_images:
        print(f'{class_name}: already has {len(existing_images)} files, skipping crawl')
        return

    print(f'Crawling {max_images} images for {class_name}: {search_query}')
    crawler = BingImageCrawler(storage={'root_dir': str(class_dir)})
    crawler.crawl(keyword=search_query, max_num=max_images, file_idx_offset=0)

for class_name, query in SELECTED_CLASSES.items():
    crawl_images_for_class(class_name, query, IMAGES_PER_CLASS)

print('Web crawling step completed.')

## 4. Clean Images And Keep 2,000 Valid Images Per Class

In [ ]:
def is_valid_image(image_path):
    try:
        with Image.open(image_path) as img:
            img.verify()
        with Image.open(image_path) as img:
            img.convert('RGB')
        return True
    except Exception:
        return False

records = []
summary_rows = []

for class_name in SELECTED_CLASSES.keys():
    class_dir = RAW_DIR / class_name
    image_paths = sorted([p for p in class_dir.glob('*') if p.is_file()])

    valid_paths = []
    invalid_paths = []

    for image_path in image_paths:
        if is_valid_image(image_path):
            valid_paths.append(image_path)
        else:
            invalid_paths.append(image_path)

    valid_paths = valid_paths[:IMAGES_PER_CLASS]

    for image_path in valid_paths:
        records.append({'path': image_path, 'class_name': class_name})

    summary_rows.append({
        'class_name': class_name,
        'downloaded_files': len(image_paths),
        'valid_images_used': len(valid_paths),
        'invalid_images': len(invalid_paths),
    })

summary_df = pd.DataFrame(summary_rows)
df = pd.DataFrame(records)
summary_df.to_csv(RESULTS_DIR / 'crawler_summary.csv', index=False)

print('Total valid images selected:', len(df))
display(summary_df)

if len(df) < TOTAL_TARGET_IMAGES:
    print('WARNING: Fewer than 10,000 valid images were collected.')
    print('Run the crawling cell again or increase crawler attempts for classes with fewer than 2,000 images.')

## 5. Create Training, Validation, And Testing Dataset Splits

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=(VALIDATION_RATIO + TEST_RATIO),
    stratify=df['class_name'],
    random_state=SEED,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=TEST_RATIO / (VALIDATION_RATIO + TEST_RATIO),
    stratify=temp_df['class_name'],
    random_state=SEED,
)

split_frames = {'train': train_df, 'validation': val_df, 'test': test_df}

for split_name in ['train', 'validation', 'test']:
    split_dir = DATASET_DIR / split_name
    if split_dir.exists():
        shutil.rmtree(split_dir)

for split_name, split_df in split_frames.items():
    for class_name in SELECTED_CLASSES.keys():
        (DATASET_DIR / split_name / class_name).mkdir(parents=True, exist_ok=True)

    for _, row in split_df.iterrows():
        source_path = Path(row['path'])
        suffix = source_path.suffix.lower() if source_path.suffix else '.jpg'
        destination = DATASET_DIR / split_name / row['class_name'] / f'{source_path.stem}{suffix}'
        shutil.copy2(source_path, destination)

    export_df = split_df.copy()
    export_df['path'] = export_df['path'].astype(str)
    export_df.to_csv(RESULTS_DIR / f'{split_name}_files.csv', index=False)

    print(split_name, len(split_df))
    print(split_df['class_name'].value_counts().sort_index())
    print()

print('Dataset split folders created in Google Drive:')
print(DATASET_DIR / 'train')
print(DATASET_DIR / 'validation')
print(DATASET_DIR / 'test')

## 6. Visualize Dataset

In [ ]:
distribution_rows = []
for split_name, split_df in split_frames.items():
    counts = split_df['class_name'].value_counts().sort_index()
    for class_name, count in counts.items():
        distribution_rows.append({'split': split_name, 'class_name': class_name, 'count': count})

distribution_df = pd.DataFrame(distribution_rows)
distribution_pivot = distribution_df.pivot(index='class_name', columns='split', values='count')
distribution_pivot = distribution_pivot[['train', 'validation', 'test']]

ax = distribution_pivot.plot(kind='bar', figsize=(10, 5))
ax.set_title('Cat Breed Dataset Split Distribution')
ax.set_xlabel('Cat Breed')
ax.set_ylabel('Number of Images')
ax.legend(title='Split')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(GRAPHS_DIR / 'class_distribution.png', dpi=150)
plt.show()

display(distribution_pivot)

In [ ]:
fig, axes = plt.subplots(len(SELECTED_CLASSES), 5, figsize=(12, 10))
for row_idx, class_name in enumerate(SELECTED_CLASSES.keys()):
    class_images = list((DATASET_DIR / 'train' / class_name).glob('*'))
    sample_images = random.sample(class_images, min(5, len(class_images)))
    for col_idx, image_path in enumerate(sample_images):
        image = Image.open(image_path).convert('RGB')
        axes[row_idx, col_idx].imshow(image)
        axes[row_idx, col_idx].axis('off')
        if col_idx == 0:
            axes[row_idx, col_idx].set_title(class_name, loc='left')
plt.tight_layout()
plt.savefig(GRAPHS_DIR / 'sample_cat_breeds.png', dpi=150)
plt.show()

## 7. Load TensorFlow Datasets

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / 'train',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=True,
    seed=SEED,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / 'validation',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False,
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / 'test',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False,
)

class_names = train_ds.class_names
num_classes = len(class_names)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

print('Classes:', class_names)

## 8. Model Training And Evaluation Functions

In [ ]:
def build_transfer_model(model_key, base_class, preprocess_fn, include_preprocessing=None):
    data_augmentation = tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.08),
        layers.RandomZoom(0.10),
    ], name=f"{model_key}_augmentation")

    base_kwargs = {
        "weights": "imagenet",
        "include_top": False,
        "input_shape": IMG_SIZE + (3,),
    }
    if include_preprocessing is not None:
        base_kwargs["include_preprocessing"] = include_preprocessing

    base_model = base_class(**base_kwargs)
    base_model.trainable = False

    inputs = layers.Input(shape=IMG_SIZE + (3,))
    x = data_augmentation(inputs)
    x = preprocess_fn(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs, name=f"{model_key}_cat_breed_classifier")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def plot_history(history_df, model_label, model_key):
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["accuracy"], label="Training Accuracy")
    plt.plot(history_df["val_accuracy"], label="Validation Accuracy")
    plt.title(f"{model_label} Training And Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(GRAPHS_DIR / f"{model_key}_accuracy.png", dpi=150)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["loss"], label="Training Loss")
    plt.plot(history_df["val_loss"], label="Validation Loss")
    plt.title(f"{model_label} Training And Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(GRAPHS_DIR / f"{model_key}_loss.png", dpi=150)
    plt.show()


def evaluate_model(model, model_label, model_key, history, training_time_seconds):
    history_df = pd.DataFrame(history.history)
    history_df.to_csv(RESULTS_DIR / f"{model_key}_history.csv", index=False)
    plot_history(history_df, model_label, model_key)

    test_loss, test_accuracy = model.evaluate(test_ds)

    y_true = []
    y_prob = []
    for images, labels in test_ds:
        predictions = model.predict(images, verbose=0)
        y_prob.append(predictions)
        y_true.append(labels.numpy())

    y_true = np.vstack(y_true)
    y_prob = np.vstack(y_prob)
    y_true_labels = np.argmax(y_true, axis=1)
    y_pred_labels = np.argmax(y_prob, axis=1)

    accuracy_from_predictions = accuracy_score(y_true_labels, y_pred_labels)
    mean_average_precision = average_precision_score(y_true, y_prob, average="macro")

    cm = confusion_matrix(y_true_labels, y_pred_labels)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(f"{model_label} Confusion Matrix")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.xticks(rotation=30, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(CONFUSION_DIR / f"{model_key}_confusion_matrix.png", dpi=150)
    plt.show()

    report = classification_report(y_true_labels, y_pred_labels, target_names=class_names, output_dict=True)
    pd.DataFrame(report).transpose().to_csv(RESULTS_DIR / f"{model_key}_classification_report.csv")

    trainable_params = int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))
    non_trainable_params = int(np.sum([np.prod(v.shape) for v in model.non_trainable_weights]))
    total_params = trainable_params + non_trainable_params

    metrics = {
        "model": model_label,
        "dataset": "Web-crawled cat breed dataset - 10,000 images",
        "classes": class_names,
        "epochs_requested": EPOCHS,
        "epochs_completed": len(history_df),
        "test_loss": float(test_loss),
        "test_accuracy": float(test_accuracy),
        "test_accuracy_from_predictions": float(accuracy_from_predictions),
        "mean_average_precision_macro": float(mean_average_precision),
        "training_time_seconds": float(training_time_seconds),
        "training_time_minutes": float(training_time_seconds / 60),
        "total_params": total_params,
        "trainable_params": trainable_params,
        "non_trainable_params": non_trainable_params,
    }

    with open(RESULTS_DIR / f"{model_key}_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    return metrics


def train_and_evaluate(model_key, model_label, base_class, preprocess_fn, include_preprocessing=None):
    print(f"Training {model_label}")
    tf.keras.backend.clear_session()
    model = build_transfer_model(model_key, base_class, preprocess_fn, include_preprocessing)
    model.summary()

    callbacks = [
        ModelCheckpoint(
            filepath=MODELS_DIR / f"{model_key}_cat_breed.keras",
            monitor="val_accuracy",
            save_best_only=True,
            mode="max",
            verbose=1,
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.3,
            patience=5,
            min_lr=1e-7,
            verbose=1,
        ),
    ]

    start_time = time.time()
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)
    training_time_seconds = time.time() - start_time

    return evaluate_model(model, model_label, model_key, history, training_time_seconds)

## 9. Train ResNet50

In [ ]:
resnet50_metrics = train_and_evaluate(
    model_key="resnet50",
    model_label="ResNet50",
    base_class=ResNet50,
    preprocess_fn=resnet50_preprocess,
)

## 10. Train DenseNet121

In [ ]:
densenet121_metrics = train_and_evaluate(
    model_key="densenet121",
    model_label="DenseNet121",
    base_class=DenseNet121,
    preprocess_fn=densenet_preprocess,
)

## 11. Train MobileNetV3Small

In [ ]:
mobilenetv3_metrics = train_and_evaluate(
    model_key="mobilenetv3",
    model_label="MobileNetV3Small",
    base_class=MobileNetV3Small,
    preprocess_fn=mobilenetv3_preprocess,
    include_preprocessing=False,
)

## 12. Final Model Comparison

In [ ]:
all_metrics = [resnet50_metrics, densenet121_metrics, mobilenetv3_metrics]

comparison_df = pd.DataFrame([
    {
        "Model": item["model"],
        "Test Accuracy": item["test_accuracy"],
        "mAP": item["mean_average_precision_macro"],
        "Training Time Minutes": item["training_time_minutes"],
        "Total Parameters": item["total_params"],
        "Epochs Completed": item["epochs_completed"],
    }
    for item in all_metrics
])

comparison_df.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)
comparison_df

In [ ]:
display_df = comparison_df.set_index("Model")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

display_df["Test Accuracy"].plot(kind="bar", ax=axes[0], color="#3b82f6")
axes[0].set_title("Test Accuracy")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=30)

display_df["mAP"].plot(kind="bar", ax=axes[1], color="#10b981")
axes[1].set_title("Mean Average Precision")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=30)

display_df["Training Time Minutes"].plot(kind="bar", ax=axes[2], color="#f97316")
axes[2].set_title("Training Time Minutes")
axes[2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(GRAPHS_DIR / "model_comparison.png", dpi=150)
plt.show()

## 13. Final Conclusion Draft

In [ ]:
best_accuracy = comparison_df.sort_values("Test Accuracy", ascending=False).iloc[0]
best_map = comparison_df.sort_values("mAP", ascending=False).iloc[0]
fastest = comparison_df.sort_values("Training Time Minutes", ascending=True).iloc[0]

conclusion = f"""
Final Conclusion:

For the cat breed classification task using the web-crawled 10,000-image dataset, the model with the highest test accuracy was {best_accuracy['Model']} with an accuracy of {best_accuracy['Test Accuracy']:.4f}.
The model with the highest mean average precision was {best_map['Model']} with an mAP of {best_map['mAP']:.4f}.
The fastest model to train was {fastest['Model']}, with a training time of {fastest['Training Time Minutes']:.2f} minutes.

The final best model should be chosen by considering accuracy, mAP, training time, parameter count, and confusion matrix results together.
If one model has slightly lower accuracy but trains much faster and has fewer parameters, it may be more suitable for practical use.
"""

print(conclusion)

with open(RESULTS_DIR / "final_conclusion.txt", "w") as f:
    f.write(conclusion)

## 14. Graph Results Preview For GitHub

Run this section after training. It displays the saved result images inside the notebook so they will appear in the GitHub notebook preview after you save the executed notebook back to GitHub.

### Dataset Graphs

In [ ]:
dataset_graphs = [
    GRAPHS_DIR / "class_distribution.png",
    GRAPHS_DIR / "sample_cat_breeds.png",
]

for graph_path in dataset_graphs:
    if graph_path.exists():
        print(graph_path.name)
        display(DisplayImage(filename=str(graph_path)))
    else:
        print("Missing:", graph_path)

### Accuracy And Loss Graphs

In [ ]:
training_graphs = [
    GRAPHS_DIR / "resnet50_accuracy.png",
    GRAPHS_DIR / "resnet50_loss.png",
    GRAPHS_DIR / "densenet121_accuracy.png",
    GRAPHS_DIR / "densenet121_loss.png",
    GRAPHS_DIR / "mobilenetv3_accuracy.png",
    GRAPHS_DIR / "mobilenetv3_loss.png",
]

for graph_path in training_graphs:
    if graph_path.exists():
        print(graph_path.name)
        display(DisplayImage(filename=str(graph_path)))
    else:
        print("Missing:", graph_path)

### Confusion Matrices

In [ ]:
confusion_matrix_graphs = [
    CONFUSION_DIR / "resnet50_confusion_matrix.png",
    CONFUSION_DIR / "densenet121_confusion_matrix.png",
    CONFUSION_DIR / "mobilenetv3_confusion_matrix.png",
]

for graph_path in confusion_matrix_graphs:
    if graph_path.exists():
        print(graph_path.name)
        display(DisplayImage(filename=str(graph_path)))
    else:
        print("Missing:", graph_path)

### Final Model Comparison Graph

In [ ]:
comparison_graph = GRAPHS_DIR / "model_comparison.png"

if comparison_graph.exists():
    display(DisplayImage(filename=str(comparison_graph)))
else:
    print("Missing:", comparison_graph)

comparison_df